# 06 — Agentic Pipeline: End-to-End Demo

Demonstrates the full Milestone 2 pipeline step-by-step using the same model/vectorizer from Milestone 1.

```
Input (text/URL)
  ↓ Step 1: ML Prediction (Milestone 1 model)
  ↓ Step 2: Risk Analysis (heuristic signals)
  ↓ Step 3: RAG Retrieval (FAISS knowledge base)
  ↓ Step 4: Uncertainty Evaluation
  ↓ Step 5: LLM Report Generation (HF API / rule-based fallback)
Structured JSON Report
```

## 1. Setup

In [ ]:
import sys, json
sys.path.insert(0, '../backend')

from agent.predictor import predict
from agent.risk_analyzer import analyze_risk
from agent.retriever import retrieve
from agent.llm_agent import run_agent

print('All modules loaded successfully')

## 2. Test Articles

In [ ]:
# Article A — likely Fake News (matches WELFake fake patterns)
FAKE_ARTICLE = """
BREAKING: SHOCKING BOMBSHELL! Obama's secret plan EXPOSED — they don't want you to know!
Deep state operatives have been caught covering up a massive conspiracy that threatens every
American. Sources close to the situation confirm that radical elements within the government
have been working to destroy our democracy. This outrageous scandal proves what patriots have
been saying for years. Share this before it gets deleted! The mainstream media is LYING to you.
Wake up America before it's too late! This is the biggest cover-up in history!!!
""".strip()

# Article B — likely Real News (matches WELFake real patterns)
REAL_ARTICLE = """
WASHINGTON (Reuters) - The Federal Reserve held interest rates steady on Wednesday, as
officials said they needed more evidence that inflation was sustainably moving toward the
2 percent target before cutting borrowing costs. The decision was unanimous, according to
a statement released after the two-day policy meeting. Fed Chair Jerome Powell said in a
press conference that the central bank remains data-dependent and will adjust policy as
economic conditions evolve. The federal funds rate remains in the 5.25-5.50 percent range,
where it has been since July 2023, the longest pause in decades.
""".strip()

print('Test articles ready.')
print(f'Fake article: {len(FAKE_ARTICLE.split())} words')
print(f'Real article: {len(REAL_ARTICLE.split())} words')

## 3. Step 1 — ML Prediction (Milestone 1 Model)

In [ ]:
print('=== FAKE ARTICLE ===')
pred_fake = predict(FAKE_ARTICLE)
for k, v in pred_fake.items():
    if k != 'cleaned_text':
        print(f'  {k}: {v}')

print('\n=== REAL ARTICLE ===')
pred_real = predict(REAL_ARTICLE)
for k, v in pred_real.items():
    if k != 'cleaned_text':
        print(f'  {k}: {v}')

## 4. Step 2 — Risk Analysis

In [ ]:
import pandas as pd

risk_fake = analyze_risk(FAKE_ARTICLE)
risk_real = analyze_risk(REAL_ARTICLE)

comparison = pd.DataFrame({
    'Metric': ['risk_score', 'clickbait_hits', 'emotional_hits', 'credibility_hits', 'caps_ratio'],
    'Fake Article': [risk_fake[k] for k in ['risk_score','clickbait_hits','emotional_hits','credibility_hits','caps_ratio']],
    'Real Article': [risk_real[k] for k in ['risk_score','clickbait_hits','emotional_hits','credibility_hits','caps_ratio']],
})
print(comparison.to_string(index=False))

print('\nFake risk factors:', risk_fake['risk_factors'])
print('Real credibility indicators:', risk_real['credibility_indicators'])

## 5. Step 3 — RAG Retrieval

In [ ]:
query_fake = f"{pred_fake['label']} {' '.join(pred_fake['top_features'][:5])}"
query_real = f"{pred_real['label']} {' '.join(pred_real['top_features'][:5])}"

docs_fake = retrieve(query_fake, top_k=3)
docs_real = retrieve(query_real, top_k=3)

print('=== Retrieved for FAKE article ===')
for d in docs_fake:
    print(f'  [{d["source"]}] {d["title"]} (score: {d.get("relevance_score","N/A")})')

print('\n=== Retrieved for REAL article ===')
for d in docs_real:
    print(f'  [{d["source"]}] {d["title"]} (score: {d.get("relevance_score","N/A")})')

## 6. Step 4 — Uncertainty Evaluation

In [ ]:
def evaluate_uncertainty(prediction: dict, risk: dict) -> dict:
    """Explicit uncertainty evaluation before report generation."""
    flags = []
    if prediction['confidence_tier'] == 'low':
        flags.append('Low model confidence — result unreliable')
    if prediction['word_count'] < 50:
        flags.append('Short article — limited signal for model')
    if risk['credibility_hits'] == 0 and risk['clickbait_hits'] == 0:
        flags.append('Ambiguous signals — neither credible nor clickbait markers found')
    return {
        'is_uncertain': len(flags) > 0,
        'uncertainty_flags': flags,
        'recommendation': 'Treat with caution' if flags else 'Proceed with analysis',
    }

unc_fake = evaluate_uncertainty(pred_fake, risk_fake)
unc_real = evaluate_uncertainty(pred_real, risk_real)

print('Fake article uncertainty:', unc_fake)
print('Real article uncertainty:', unc_real)

## 7. Step 5 — Full Agent Run (Report Generation)

In [ ]:
print('Running agent on FAKE article...')
state_fake = run_agent(FAKE_ARTICLE, pred_fake, risk_fake, docs_fake)

print(f'Steps completed: {state_fake["steps_completed"]}')
print(f'Used LLM: {state_fake["used_llm"]}')
print()
print(json.dumps(state_fake['report'], indent=2))

In [ ]:
print('Running agent on REAL article...')
state_real = run_agent(REAL_ARTICLE, pred_real, risk_real, docs_real)

print(f'Steps completed: {state_real["steps_completed"]}')
print(f'Used LLM: {state_real["used_llm"]}')
print()
print(json.dumps(state_real['report'], indent=2))

## 8. Full Pipeline via API (live server test)

In [ ]:
# Requires: uvicorn agent_app:app --port 8001 running in backend/
import requests as req

try:
    resp = req.post(
        'http://127.0.0.1:8001/analyze',
        json={'text': FAKE_ARTICLE},
        timeout=30
    )
    if resp.status_code == 200:
        data = resp.json()
        print('API Response:')
        print(f'  Label: {data["prediction"]["label"]}')
        print(f'  Confidence: {data["prediction"]["confidence"]}%')
        print(f'  Risk Score: {data["risk_analysis"]["risk_score"]}/100')
        print(f'  Summary: {data["report"]["summary"]}')
    else:
        print(f'API error: {resp.status_code} — {resp.text}')
except Exception as e:
    print(f'Server not running or error: {e}')
    print('Start with: cd backend && uvicorn agent_app:app --port 8001 --reload')

## Pipeline Summary

| Step | Module | Output |
|---|---|---|
| 1. ML Prediction | `agent/predictor.py` | label, confidence, top features |
| 2. Risk Analysis | `agent/risk_analyzer.py` | risk score, risk factors, credibility indicators |
| 3. RAG Retrieval | `agent/retriever.py` | top-3 relevant KB documents |
| 4. Uncertainty | inline evaluation | uncertainty flags |
| 5. Report | `agent/llm_agent.py` | structured JSON report |